In [ ]:
import io
import os
import re
import html
import difflib
import json
from pathlib import Path
from IPython.display import HTML, display, Audio
from openai import OpenAI
from dotenv import load_dotenv
from pydub import AudioSegment
from mutagen.mp3 import MP3

audio_path = Path("HI009clip.mp3")

assert audio_path.is_file(), "Audio file not found!"

In [2]:
audio = MP3(audio_path)

print(f"Duration: {audio.info.length:.2f} seconds")
print(f"Sample Rate: {audio.info.sample_rate} Hz")
print(f"Channels: {audio.info.channels}")
print(f"Bitrate: {audio.info.bitrate / 1000:.0f} kbps")

display(Audio(filename=str(audio_path)))

Duration: 181.53 seconds
Sample Rate: 44100 Hz
Channels: 1
Bitrate: 89 kbps


In [ ]:
# Load original audio
audio = AudioSegment.from_file(audio_path)

# Extract first 30 seconds
short_audio = audio[:30_000]

# Create in-memory WAV file
buffer = io.BytesIO()
short_audio.export(buffer, format="wav")
buffer.seek(0)
buffer.name = "short_audio.wav"

print("Audio shortened to 30 seconds.")

Audio shortened to 30 seconds.


In [ ]:
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Transcribe with Whisper
buffer.seek(0)

basic_transcript = client.audio.transcriptions.create(
    model="whisper-1",
    file=buffer
)

# Extract transcription text
print("\n📝 Transcription:")
print("-" * 40)
print(basic_transcript.text)

API Response:
Transcription(text="I hear that there's a Hawaiian custom of something called a hukilau. Hukilau, hukilau is they have a rope, end of the rope they have a net there. They surround that certain place and they drag the net. They surround it, you mean they carry the net out? Carry the net out. How do they, is there something to keep the net floating? Some have those floaters floating.", languages=None, logprobs=None, usage=UsageDuration(seconds=30.0, type='duration'))

📝 Transcription:
----------------------------------------
I hear that there's a Hawaiian custom of something called a hukilau. Hukilau, hukilau is they have a rope, end of the rope they have a net there. They surround that certain place and they drag the net. They surround it, you mean they carry the net out? Carry the net out. How do they, is there something to keep the net floating? Some have those floaters floating.


In [ ]:
# Save unguided transcription from Step 3
text_without_context = basic_transcript.text

# Transcribe with context prompt
buffer.seek(0)

transcript_with_context = client.audio.transcriptions.create(
    model="whisper-1",
    file=buffer,
    prompt="Hukilau, Hawaiian fishing tradition, fishing nets, ropes, floats."
)

# Save for comparison in Step 5
text_with_context = transcript_with_context.text

print("\n📝 Transcription with context:")
print("-" * 40)
print(text_with_context)


📝 Transcription with context:
----------------------------------------
I hear that there's a Hawaiian custom of something called a hukilau. Hukilau, hukilau is they have a rope, end of the rope they have a net there. They surround that certain place and they drag the net. They surround it, you mean they carry the net out? Carry the net, uh-huh. How do they, is there something to keep the net floating? Yeah, sometimes those floaters are floating.


In [ ]:
# Reference transcription (first 30 seconds)
reference = """
Uh, I hear that there’s a Hawaiian custom of something called a hukilau.
Hukilau, hukilau is, uh, they have a rope, end of the rope they have a net there.
Mm-hmm.
They surround it, a certain place, and they drag the net.
They surround it. You mean they carry the net out?
Carry it, yeah, uh, huh.
Uh, how do they, i-, is there something to keep the net floating?
Yeah, sometimes those, uh, floaters, uh, floating.
""".strip()

def tokenize(text):
    """Split text into words, keeping numbers like 7.2 together."""
    return re.findall(r"\d+(?:\.\d+)?|[\w'’-]+", text)

def word_error_rate(reference, hypothesis):
    """Share of words that are wrong, missing or extra compared to the reference."""
    ref = [w.lower() for w in tokenize(reference)]
    hyp = [w.lower() for w in tokenize(hypothesis)]

    d = [[0] * (len(hyp) + 1) for _ in range(len(ref) + 1)]

    for i in range(len(ref) + 1):
        d[i][0] = i
    for j in range(len(hyp) + 1):
        d[0][j] = j

    for i in range(1, len(ref) + 1):
        for j in range(1, len(hyp) + 1):
            cost = 0 if ref[i - 1] == hyp[j - 1] else 1
            d[i][j] = min(
                d[i - 1][j] + 1,         # Missing word
                d[i][j - 1] + 1,         # Extra word
                d[i - 1][j - 1] + cost  # Same or wrong word
            )

    return d[len(ref)][len(hyp)] / len(ref)

# Calculate Word Error Rate
wer_before = word_error_rate(reference, text_without_context)
wer_after = word_error_rate(reference, text_with_context)

# Compare both transcriptions
before = tokenize(text_without_context)
after = tokenize(text_with_context)

rows = ""

for op, i1, i2, j1, j2 in difflib.SequenceMatcher(
    None,
    [w.lower() for w in before],
    [w.lower() for w in after]
).get_opcodes():
    if op != "equal":
        old = html.escape(" ".join(before[i1:i2])) or "—"
        new = html.escape(" ".join(after[j1:j2])) or "—"
        rows += f"<tr><td style='color:#c0392b'>{old}</td><td style='color:#1e8449'>{new}</td></tr>"

display(HTML(f"""
<h3>Word Error Rate (lower is better)</h3>
<p>Without context: <b>{wer_before:.1%}</b> &nbsp;→&nbsp; With context: <b>{wer_after:.1%}</b></p>
<h3>Where the two transcripts differ</h3>
<table>
  <tr><th>Without context</th><th>With context</th></tr>
  {rows or "<tr><td colspan='2'>No differences</td></tr>"}
</table>
"""))

Without context,With context
out,uh-huh
Some have,Yeah sometimes
—,are


In [ ]:
# Load new audio file
meeting_path = Path("project_nimbus_meeting.mp3")
assert meeting_path.is_file(), "Audio file not found!"

audio = AudioSegment.from_file(meeting_path)

# Define duration and chunk size
duration = 120        # Seconds of audio to process
chunk_duration = 20   # Seconds per chunk

# Extract first 120 seconds
audio = audio[:duration * 1000]

# Split audio into chunks
chunk_size = chunk_duration * 1000
chunks = [audio[i:i+chunk_size] for i in range(0, len(audio), chunk_size)]

# Export chunks
chunk_dir = Path("chunks")
chunk_dir.mkdir(exist_ok=True)

chunk_paths = []
chunk_offsets = []

for i, chunk in enumerate(chunks):
    chunk_path = chunk_dir / f"chunk_{i+1}.wav"
    chunk.export(chunk_path, format="wav")

    chunk_paths.append(chunk_path)
    chunk_offsets.append(i * chunk_duration)

    print(f"Chunk {i+1}: {len(chunk)/1000:.1f} seconds")

print(f"\n✅ Split into {len(chunks)} chunks")

Chunk 1: 20.0 seconds
Chunk 2: 20.0 seconds
Chunk 3: 20.0 seconds
Chunk 4: 20.0 seconds
Chunk 5: 20.0 seconds
Chunk 6: 20.0 seconds

✅ Split into 6 chunks


In [ ]:
# Transcribe chunks with timestamps
all_transcripts = []
all_segments = []
all_words = []

for i, (chunk_path, offset) in enumerate(zip(chunk_paths, chunk_offsets), start=1):
    print(f"🤖 Transcribing chunk {i}/{len(chunk_paths)}...")

    chunk_transcript = client.audio.transcriptions.create(
        model="whisper-1",
        file=chunk_path,
        response_format="verbose_json",
        timestamp_granularities=["segment", "word"]
    )

    all_transcripts.append(chunk_transcript.text)

    # Adjust segment timestamps
    for segment in chunk_transcript.segments or []:
        all_segments.append({
            "start": segment.start + offset,
            "end": segment.end + offset,
            "text": segment.text.strip()
        })

    # Adjust word timestamps
    for word in chunk_transcript.words or []:
        all_words.append({
            "start": word.start + offset,
            "end": word.end + offset,
            "word": word.word
        })

# Combine transcriptions
full_text = " ".join(all_transcripts)

# Display transcription with timestamps
print("\n📝 Complete Transcription with Timestamps:")
print("-" * 40)

for segment in all_segments:
    print(f"[{segment['start']:.2f}s - {segment['end']:.2f}s] {segment['text']}")

🤖 Transcribing chunk 1/6...
🤖 Transcribing chunk 2/6...
🤖 Transcribing chunk 3/6...
🤖 Transcribing chunk 4/6...
🤖 Transcribing chunk 5/6...
🤖 Transcribing chunk 6/6...

📝 Complete Transcription with Timestamps:
----------------------------------------
[0.00s - 6.14s] Good morning everyone. This is the weekly engineering sync for Project Nimbus. Today
[6.14s - 13.06s] is Thursday, and we have four items on the agenda. The Kubernetes migration, the PostgreSQL
[13.06s - 19.98s] latency issue, the Q3 OKRs, and the hiring update. Priya, do you want to start?
[20.00s - 20.92s] with the migration.
[22.46s - 24.10s] Sure, thanks Marcus.
[24.60s - 28.54s] So the kubernetes migration is about 70% done.
[28.94s - 31.32s] We moved the authentication service,
[31.52s - 35.50s] and the billing service to the new EKS cluster last week.
[35.58s - 38.14s] The main blocker right now is the RIDIZ cache.
[38.58s - 39.98s] When we deployed it with...
[40.00s - 45.96s] The persistent volume claims were fail

In [ ]:
# Create export directory
export_dir = Path("exports")
export_dir.mkdir(exist_ok=True)

# Format timestamps
def format_timestamp(seconds, separator="."):
    milliseconds = round(seconds * 1000)
    hours, remainder = divmod(milliseconds, 3600000)
    minutes, remainder = divmod(remainder, 60000)
    secs, ms = divmod(remainder, 1000)
    return f"{hours:02}:{minutes:02}:{secs:02}{separator}{ms:03}"

# Export TXT with timestamps
txt_path = export_dir / "transcription.txt"

txt_content = "\n".join(
    f"[{format_timestamp(s['start'])} - {format_timestamp(s['end'])}] {s['text']}"
    for s in all_segments
)

txt_path.write_text(txt_content, encoding="utf-8")

# Export SRT subtitles
srt_path = export_dir / "transcription.srt"

srt_content = "\n\n".join(
    f"{i}\n"
    f"{format_timestamp(s['start'], ',')} --> {format_timestamp(s['end'], ',')}\n"
    f"{s['text']}"
    for i, s in enumerate(all_segments, start=1)
)

srt_path.write_text(srt_content + "\n", encoding="utf-8")

# Export JSON with timestamps and metadata
json_path = export_dir / "transcription.json"

data = {
    "source_file": meeting_path.name,
    "processed_duration_seconds": len(audio) / 1000,
    "chunk_duration_seconds": chunk_duration,
    "text": full_text,
    "segments": all_segments,
    "words": all_words
}

json_path.write_text(
    json.dumps(data, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

# Verify exported files
for path in [txt_path, srt_path, json_path]:
    assert path.is_file() and path.stat().st_size > 0
    path.read_text(encoding="utf-8")
    print(f"✅ Exported: {path}")

# Verify JSON structure
json.loads(json_path.read_text(encoding="utf-8"))

print("\n✅ All exports completed successfully!")

✅ Exported: exports\transcription.txt
✅ Exported: exports\transcription.srt
✅ Exported: exports\transcription.json

✅ All exports completed successfully!
